In [1]:
import json
import os
import torch
import torch.utils.data as data
from PIL import Image
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="9RjAcfQ7qNOdxfJS7iIH")
project = rf.workspace("work-wwys1").project("dogs-hrmkk")
version = project.version(1)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...


In [3]:
class DogDataset(data.Dataset):
    def __init__(self, path, folder, transform=None):
        self.path = os.path.join(path, folder)
        self.transform = transform
        
        with open("format.json", "r") as fp:
            self.format = json.load(fp)
            
            self.length = 0
            self.files = []
            
            for _dir, _target in self.format.items():
                path = os.path.join(self.path, _dir)
                list_files = os.listdir(path)
                self.length += len(list_files)
                
                for file in list_files:
                    self.files.append((os.path.join(path, file), _target))

    def __getitem__(self, item):
        path_file, target = self.files[item]
        img = Image.open(path_file)
        
        if self.transform:
            img = self.transform(img)
        
        return img, target

    def __len__(self):
        return self.length

In [4]:
model = models.resnet50(weights=None)
resnet_weights = "resnet50-0676ba61.pth"
state_dict = torch.load(resnet_weights)
model.load_state_dict(state_dict)

model_param = models.ResNet50_Weights.DEFAULT
transforms = model_param.transforms()

model.requires_grad_(False)

model.fc = nn.Sequential(
    nn.Linear(2048, 1024, bias=False),
    nn.BatchNorm1d(1024),
    nn.ReLU(inplace=True),
    nn.Linear(1024, 512, bias=False),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Linear(512, 10)
)

model.fc.requires_grad_(True)

d_train = DogDataset("dogs-1", "train", transform=transforms)
train_data = data.DataLoader(d_train, batch_size=64, shuffle=True)

d_val = DogDataset("dogs-1", "valid", transform=transforms)
valid_data = data.DataLoader(d_val, batch_size=64, shuffle=False)

d_test = DogDataset("dogs-1", "test", transform=transforms)
test_data = data.DataLoader(d_test, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = optim.Adam(params=model.fc.parameters(), lr=0.001, weight_decay=0.001)
loss_function = nn.CrossEntropyLoss()
epochs = 50
best_val_loss = float('inf')
patience = 10
patience_counter = 0

In [5]:
for _e in range(epochs):
    loss_mean = 0
    lm_count = 0
    loss_val_mean = 0
    lm_val_count = 0
    
    model.train()
    train_tqdm = tqdm(train_data, leave=True, colour="blue")
    for x_train, y_train in train_tqdm:
        x_train, y_train = x_train.to(device), y_train.to(device)
        predict = model(x_train)
        
        loss = loss_function(predict, y_train)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss_mean={loss_mean: .3f}")
    
    model.eval()
    val_tqdm = tqdm(valid_data, leave=True, colour="green")
    with torch.no_grad():
        for x_val, y_val in val_tqdm:
            x_val, y_val = x_val.to(device), y_val.to(device)
            predict_val = model(x_val)
            loss_val = loss_function(predict_val, y_val)
            lm_val_count += 1
            loss_val_mean = 1/lm_val_count * loss_val.item() + (1 - 1/lm_val_count) * loss_val_mean
            val_tqdm.set_description(f"Epoch [{_e+1}/{epochs}], loss_val_mean={loss_val_mean: .3f}")
        
        if loss_val_mean < best_val_loss:
            best_val_loss = loss_val_mean
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f'Процесс прекращен на эпохе {_e + 1}')
            break

Q = 0
P = 0
count = 0
best_coefficients = torch.load("best_model.pth", weights_only=True)
model.load_state_dict(best_coefficients)
model.eval()

test_tqdm = tqdm(test_data, leave=True, colour="red")
for x_test, y_test in test_tqdm:
    x_test, y_test = x_test.to(device), y_test.to(device)
    with torch.no_grad():
        p = model(x_test)
        p2 = torch.argmax(p, dim=1)
        P += torch.sum(p2 == y_test).item()
        Q += loss_function(p, y_test).item()
        count += 1

Q /= count
P /= len(d_test)
print(f"Q = {Q}")
print(f"P = {P}")

Epoch [12/50], loss_val_mean= 0.334: 100%|██████████| 6/6 [00:02<00:00,  2.14it/s]


Процесс прекращен на эпохе 12


100%|██████████| 3/3 [00:01<00:00,  1.97it/s]

Q = 0.1821655829747518
P = 0.953125
